In [1]:
import math

def calculate_hx_outlets(m_h, cp_h, T_h_in, m_c, cp_c, T_c_in, UA):
    """
    Calculates the outlet temperatures of a counter-flow double-pipe heat exchanger
    using the Effectiveness-NTU method.
    """
    # 1. Calculate heat capacity rates
    C_h = m_h * cp_h
    C_c = m_c * cp_c
    
    # 2. Determine min and max fluids
    if C_h < C_c:
        C_min = C_h
        C_max = C_c
        min_fluid = "Hot Fluid"
    else:
        C_min = C_c
        C_max = C_h
        min_fluid = "Cold Fluid"
        
    C = C_min / C_max
    
    # 3. Calculate NTU
    NTU = UA / C_min
    
    # 4. Calculate Effectiveness (epsilon)
    # Using math.isclose to handle floating point precision when C is practically 1
    if math.isclose(C, 1.0, rel_tol=1e-5):
        epsilon = NTU / (NTU + 1)
    else:
        epsilon = (1 - math.exp(-NTU * (1 - C))) / (1 - C * math.exp(-NTU * (1 - C)))
        
    # 5. Calculate Heat Transfer
    q_max = C_min * (T_h_in - T_c_in)
    q_actual = epsilon * q_max
    
    # 6. Calculate Outlet Temperatures
    # Using the universal energy balance 
    T_h_out = T_h_in - (q_actual / C_h)
    T_c_out = T_c_in + (q_actual / C_c)
    
    # Compile results into a dictionary for easy extraction
    results = {
        "T_h_out": round(T_h_out, 2),
        "T_c_out": round(T_c_out, 2),
        "q_actual": round(q_actual, 2),
        "epsilon": round(epsilon, 4),
        "NTU": round(NTU, 4),
        "C_ratio": round(C, 4),
        "min_fluid": min_fluid
    }
    
    return results

# ==========================================
# Example Usage:
# ==========================================
if __name__ == "__main__":
    # Example inputs (Standard Water-Water setup)
    inputs = {
        "m_h": 0.05,       # kg/s
        "cp_h": 4178,      # J/(kg*K)
        "T_h_in": 60.0,    # deg C
        "m_c": 0.033,      # kg/s
        "cp_c": 4178,      # J/(kg*K)
        "T_c_in": 20.0,    # deg C
        "UA": 200.0        # W/K
    }

    # Run the calculator
    output = calculate_hx_outlets(**inputs)

    # Display results
    print("--- Heat Exchanger Results ---")
    print(f"Minimum Fluid:     {output['min_fluid']}")
    print(f"Capacity Ratio (C): {output['C_ratio']}")
    print(f"NTU:               {output['NTU']}")
    print(f"Effectiveness:     {output['epsilon']}")
    print(f"Actual Heat Xfer:  {output['q_actual']} W")
    print(f"Hot Outlet Temp:   {output['T_h_out']} °C")
    print(f"Cold Outlet Temp:  {output['T_c_out']} °C")

--- Heat Exchanger Results ---
Minimum Fluid:     Cold Fluid
Capacity Ratio (C): 0.66
NTU:               1.4506
Effectiveness:     0.6522
Actual Heat Xfer:  3596.82 W
Hot Outlet Temp:   42.78 °C
Cold Outlet Temp:  46.09 °C
